# 06 — GNN: Graph Features for Ring Detection (L_R)

Build a bipartite card↔merchant graph, train GraphSAGE,
extract scalar features, and test via LR test whether they
add significant information beyond the classical GLM.

**Requires:** `pip install torch torch-geometric`

**Primary label target:** `L_R` (ring membership)  
**Nested test:** M_classical ⊂ M_{+GNN}

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
sys.path.insert(0, '..')

from src.labels import LABEL_COLS
from src.features import build_features, label_matrix
from src.models.gnn import build_bipartite_graph, CardMerchantSAGE
from src.evaluation import lr_test, multi_label_report

In [ ]:
train = pd.read_parquet('../data/processed/train_labeled.parquet')
test  = pd.read_parquet('../data/processed/test_labeled.parquet')

X_train = build_features(train)
X_test  = build_features(test)
y_train = label_matrix(train)
y_test  = label_matrix(test)

In [ ]:
# Build graph on training data
data, card_idx, merch_idx = build_bipartite_graph(train)
print(f'Card nodes: {len(card_idx):,} | Merchant nodes: {len(merch_idx):,}')
print(f'Edges: {data["card", "transacts", "merchant"].edge_index.shape[1]:,}')

In [ ]:
# Train GraphSAGE
sage = CardMerchantSAGE(in_dim=2, hid=64, out_dim=32, epochs=50)
sage.fit(data)

In [ ]:
# Extract per-transaction scalar features
gnn_feats_train = sage.extract_features(data, train, card_idx)
gnn_feats_test  = sage.extract_features(data, test,  card_idx)
print(gnn_feats_train.describe())

In [ ]:
# LR test per label — does GNN help?
from src.models.glm import BinaryRelevanceGLM

with open('../data/processed/baseline_log_likelihoods.json') as f:
    baseline_lls = json.load(f)

br_gnn = BinaryRelevanceGLM()
for label in LABEL_COLS:
    result = br_gnn.admit_extension(X_train, gnn_feats_train, y_train, label)
    sym = '✓ ADMITTED' if result['admitted'] else '✗ dropped'
    print(f'{label}: G²={result["G2"]:.2f}  p={result["p_value"]:.4f}  {sym}')

In [ ]:
# Save GNN features for combined model in 08_benchmark
gnn_feats_train.to_parquet('../data/processed/gnn_feats_train.parquet')
gnn_feats_test.to_parquet('../data/processed/gnn_feats_test.parquet')
print('Saved GNN features.')